# Gemini + RAG demo - reproducibility analysis of a single PDF

Uses `gemini_rag.GeminiPaperAnalyst` to extract a structured `PaperProfile` from any PDF on disk.

**Setup**

```bash
uv pip install google-genai pydantic
export GEMINI_API_KEY=your-key-here   # or GOOGLE_API_KEY
```

The pipeline runs five independent, schema-constrained queries against the PDF (methodology, datasets, artefacts, figures/tables, parameters) at `temperature=0` with a system instruction that pins answers to the attached document. Each extracted item carries a `source_quote` field so you can audit for hallucination.

## 0. Available paired papers

Lists every row of `data/rescience_bibtex_table.xlsx` that has a PDF in **both** the `RESCIENCE C` and `ORIGINAL` sections. Pick one of these filenames as the `PDF_PATH` in section 2.

In [1]:
import pandas as pd
from pathlib import Path

XLSX = Path("data/rescience_bibtex_table.xlsx")
df_all = pd.read_excel(XLSX).iloc[1:].reset_index(drop=True)

RESCIENCE_PDF_COL = 18
ORIGINAL_PDF_COL = 28
TITLE_RESCIENCE_COL = 2
TITLE_ORIGINAL_COL = 19

mask = df_all.iloc[:, RESCIENCE_PDF_COL].notna() & df_all.iloc[:, ORIGINAL_PDF_COL].notna()
paired = df_all[mask].copy()
paired_view = pd.DataFrame({
    "rescience_pdf": paired.iloc[:, RESCIENCE_PDF_COL].values,
    "original_pdf":  paired.iloc[:, ORIGINAL_PDF_COL].values,
    "rescience_title": paired.iloc[:, TITLE_RESCIENCE_COL].values,
    "original_title":  paired.iloc[:, TITLE_ORIGINAL_COL].values,
})
print(f"{len(paired_view)} papers have PDFs in both sections\n")
paired_view

26 papers have PDFs in both sections



,rescience_pdf,original_pdf,rescience_title,original_title
0,2023_04_article.pdf,2023_04_article.pdf,[Re] Object Detection Meets Knowledge Graphs,Yuan Fang; Kingsley Kuan; Jie Lin; Cheston Tan...
1,2023_18_article.pdf,2023_18_article.pdf,[Re] DialSummEval - Evaluation of automatic su...,Mingqi Gao; Xiaojun Wan
2,2023_36_article.pdf,2023_36_article.pdf,"[Re] If you like Shapley, then you'll love the...",Tom Yan; Ariel D. Procaccia
3,2023_37_article.pdf,2023_37_article.pdf,[Re] A Reproduction of Automatic Multi-Label P...,Han Wang; Canwen Xu; Julian McAuley
4,2023_48_article.pdf,2023_48_article.pdf,[Re] A Replication Study of Compositional Gene...,"Shaw, Peter and Chang, Ming-Wei and Pasupat, P..."
5,2022_12_article.pdf,2022_12_article.pdf,[Re] A Cluster-based Approach for Improving Is...,"Rajaee, Sara and Pilehvar, Mohammad Taher"
6,2022_13_article.pdf,2022_13_article.pdf,"[Re] Reproduction and Extension of ""Queens are...","Dinan, Emily and Fan, Angela and Williams, Adi..."
7,2022_14_article.pdf,2022_14_article.pdf,[Re] Reproduction Study of Variational Fair Cl...,"Ziko, Imtiaz Masud and Yuan, Jing and Granger,..."
8,2022_35_article.pdf,2022_35_article.pdf,[Re] Solving Phase Retrieval With a Learned Re...,"Hyder, Rakib and Cai, Zikui and Asif, M Salman"
9,2022_40_article.pdf,2022_40_article.pdf,[Re] Badder Seeds: Reproducing the Evaluation ...,"Antoniak, Maria and Mimno, David"


## 1. Imports and setup

In [2]:
import json
from pathlib import Path

from gemini_rag import GeminiPaperAnalyst, PaperProfile

## 2. Point at a specific PDF

Change `PDF_PATH` to any paper you want to analyse. Defaults to one of the cached originals.

In [3]:
PDF_PATH = Path("data/pdf_original_cache/2023_04_article.pdf")
assert PDF_PATH.exists(), f"missing: {PDF_PATH}"
print(f"PDF: {PDF_PATH}  ({PDF_PATH.stat().st_size / 1024:.1f} KB)")

PDF: data/pdf_original_cache/2023_04_article.pdf  (2743.7 KB)


## 3. Run the analyst

In [4]:
analyst = GeminiPaperAnalyst(model="gemini-2.5-flash")
profile: PaperProfile = analyst.analyze(PDF_PATH)

Uploading 2023_04_article.pdf to Gemini Files API...
  querying: header ...
  querying: methodology ...
  querying: datasets ...
  querying: artefacts ...
  querying: figures_tables ...
  querying: parameters ...


## 4. Inspect the PaperProfile

In [5]:
print(f"Title:   {profile.title}")
print(f"Authors: {', '.join(profile.authors)}")
print(f"Methodology steps: {len(profile.methodology_steps)}")
print(f"Datasets:          {len(profile.datasets)}")
print(f"Figures:           {len(profile.figures_to_reproduce)}")
print(f"Tables:            {len(profile.tables_to_reproduce)}")
print(f"Hyperparameters:   {len(profile.hyperparameters)}")
print(f"Repository links:  {profile.repository_links}")

Title:   Object Detection Meets Knowledge Graphs
Authors: Yuan Fang, Kingsley Kuan, Jie Lin, Cheston Tan, Vijay Chandrasekhar
Methodology steps: 4
Datasets:          3
Figures:           4
Tables:            3
Hyperparameters:   17
Repository links:  ['https://github.com/rbgirshick/py-faster-renn']


### Methodology steps (with grounding quotes)

In [6]:
for step in profile.methodology_steps:
    print(f"[{step.order}] {step.description}")
    if step.tools_mentioned:
        print(f"    tools: {', '.join(step.tools_mentioned)}")
    if step.source_quote:
        print(f'    quote: "{step.source_quote}"')
    print()

[1] An existing object detection algorithm (Faster R-CNN with VGG-16) is trained and then used to generate initial bounding box detections and their associated probabilities for each image.
    tools: Faster R-CNN, VGG-16, Python Caffe implementation (version not stated), stochastic gradient descent, momentum of 0.9, mini-batch size of 2, weight decay of 5e-4, Gaussian distribution with a standard deviation of 0.01, learning rate of 1e-3, learning rate of 1e-4
    quote: "We assume an existing object detection algorithm that outputs a set of bounding box B = {1,2,..., B} for each image, and assigns a label l∈ L to each bounding box b ∈ B with probability p(lb). We employ the state-of-the-art Faster R-CNN and VGG-16 as the baseline [Simonyan and Zisserman, 2014; Ren et al., 2015], using the public Python Caffe implementation4."

[2] Semantic consistency (S) is computed for each pair of concepts using the frequency of their co-occurrences in background data, based on pointwise mutual inf

### Datasets

In [7]:
for ds in profile.datasets:
    print(f"- {ds.name}  ({ds.availability.value})")
    if ds.url:
        print(f"    url: {ds.url}")
    if ds.source_quote:
        print(f'    quote: "{ds.source_quote}"')

- MSCOCO15  (open)
    url: http://mscoco.org/home/
    quote: "We use benchmark data MSCOCO15 [Lin et al., 2014] and PASCAL07 [Everingham et al., 2010], summarized in Ta-ble 1. For MSCOCO15, we combine their training and valida-tion sets for training the baseline, except for a subset of 5000 images named “minival”. We further split minival into 1000 and 4000 images, named “minival-1k” and “minival-4k" re-spectively. We use minival-1k to choose hyperparameter for our approach, and minival-4k for offline testing. Online eval-uation on the MSCOCO15 server³ is performed on the test set, since its ground truth is not publicly available. The test set contains two subsets of roughly equal size, namely “test-dev" and "test-std”, where the latter only allows for limited submissions. For PASCAL07, we use their training set for training the baseline, validation set for choosing our hyperpa-rameter, and test set for evaluation. http://mscoco.org/home/"
- PASCAL07  (not_mentioned)
    quote: "We u

## 5. Save the profile to JSON

In [8]:
out = PDF_PATH.with_suffix(".profile.json")
out.write_text(profile.model_dump_json(indent=2))
print(f"wrote: {out}")

wrote: data/pdf_original_cache/2023_04_article.profile.json


## 6. Token usage and cost estimate

`analyst.usage_log` captures `usage_metadata` from every call since the last `analyze()`. `analyst.usage_summary()` aggregates it and applies the paid-tier rates in `gemini_rag.GEMINI_PRICING_USD_PER_MTOK`.

Rates drift - always cross-check against https://ai.google.dev/pricing and your AI Studio billing dashboard for authoritative numbers.

In [9]:
summary = analyst.usage_summary()

print(f"Model: {summary['model']}")
print(f"API calls: {summary['calls']}")
print()
print(f"{'query':<18} {'input':>10} {'output':>10}")
print("-" * 40)
for q in summary["per_query"]:
    print(f"{q['name']:<18} {q['input_tokens']:>10,} {q['output_tokens']:>10,}")
print("-" * 40)
print(f"{'TOTAL':<18} {summary['input_tokens']:>10,} {summary['output_tokens']:>10,}")
print(f"Grand total tokens: {summary['total_tokens']:,}")
print()
if summary["total_usd"] is None:
    print(f"No price table entry for model {summary['model']!r} - add one to GEMINI_PRICING_USD_PER_MTOK.")
else:
    print(
        f"Estimated cost (USD):  input ${summary['input_usd']:.6f}  "
        f"+  output ${summary['output_usd']:.6f}  "
        f"=  ${summary['total_usd']:.6f}"
    )

Model: gemini-2.5-flash
API calls: 6

query                   input     output
----------------------------------------
header                  1,913         60
methodology             1,962      1,406
datasets                1,949      1,234
artefacts               1,936        116
figures_tables          1,932      1,139
parameters              1,932      1,286
----------------------------------------
TOTAL                  11,624      5,241
Grand total tokens: 16,865

Estimated cost (USD):  input $0.003487  +  output $0.013102  =  $0.016590
